# RACECAR Neo — Async Core Test

This notebook walks through every sensor, control surface, and driver feature exposed by the v2 student library and the `racecar_neo_ros2_driver` backend. Run each cell in order. The car must be powered on with the teleop stack already running:

```bash
ros2 launch racecar_neo_ros2_driver teleop.launch.py edgetpu_enable:=true
```

Each section runs for a fixed window (defaults to 10 s) and reports live counters. The final "Summary" cell aggregates pass/fail for every subsystem.

## 1. Initialize Racecar

In [1]:
import sys, os, time, io

import numpy as np
import cv2 as cv
from IPython.display import display
import ipywidgets as widgets

# Make the v2 library importable when this notebook is launched from labs/tests/
LIB_DIR = os.path.abspath(os.path.join(os.getcwd(), '..', '..', 'library'))
if LIB_DIR not in sys.path:
    sys.path.insert(0, LIB_DIR)

import racecar_core
import racecar_utils as rc_utils

In [2]:
def to_jpeg_bytes(bgr_image, resize=(320, 240)):
    """Convert a BGR numpy image to JPEG bytes for widget display."""
    if resize:
        bgr_image = cv.resize(bgr_image, resize)
    _, buf = cv.imencode('.jpg', bgr_image, [cv.IMWRITE_JPEG_QUALITY, 70])
    return buf.tobytes()

In [3]:
# Create the racecar in real mode and start the async ROS2 executor
rc = racecar_core.create_racecar(isSimulation=False)
rc.go_async()
print('Racecar initialized, async executor running.')
print('Waiting 3 seconds for topics to connect...')
time.sleep(3)

ModuleNotFoundError: No module named 'rclpy'

## 2. Forward Camera Stream (10 seconds)

Subscribes to `/camera/forward` (`sensor_msgs/Image`, raw BGR8). The v2 driver decodes MJPEG inside gscam, so the student library only does a `CvBridge.imgmsg_to_cv2(..., 'bgr8')`.

In [ ]:
DURATION = 10
img_widget = widgets.Image(format='jpeg', width=320, height=240)
label = widgets.Label(value='Starting...')
display(widgets.VBox([label, img_widget]))

frame_count = 0
t_start = time.monotonic()

while time.monotonic() - t_start < DURATION:
    color_image = rc.camera.get_color_image_async()
    if color_image is not None:
        frame_count += 1
        img_widget.value = to_jpeg_bytes(color_image)
        elapsed = time.monotonic() - t_start
        fps = frame_count / elapsed if elapsed > 0 else 0
        label.value = f'Forward camera | Frame {frame_count} | {elapsed:.1f}s / {DURATION}s | {fps:.1f} FPS'
    time.sleep(0.05)

elapsed = time.monotonic() - t_start
rate = frame_count / elapsed if elapsed > 0 else 0
label.value = f'Forward camera complete: {frame_count} frames in {elapsed:.1f}s = {rate:.1f} FPS'
forward_received = rc.camera.get_color_image_async() is not None

## 3. Depth API (RealSense D435i)

The forward camera is now an Intel RealSense D435i, so depth is available. `rc.camera.get_depth_image()` / `get_depth_image_async()` read the depth stream from `/camera/depth`. This cell exercises the depth path.

In [ ]:
import cv2 as cv

DURATION = 10
MAX_DEPTH_CM = 300  # 3 m, practical indoor range

img_widget = widgets.Image(format='jpeg', width=320, height=240)
label = widgets.Label(value='Starting...')
display(widgets.VBox([label, img_widget]))

frame_count = 0
t_start = time.monotonic()

while time.monotonic() - t_start < DURATION:
    depth_image = rc.camera.get_depth_image_async()
    if depth_image is not None:
        frame_count += 1
        # Close = red, far = blue (inverted); no reading (0 cm) = black.
        depth_normalized = np.clip(depth_image / MAX_DEPTH_CM, 0.0, 1.0)
        depth_display = (255 - depth_normalized * 255).astype(np.uint8)
        depth_display[depth_image == 0] = 0
        depth_colored = cv.applyColorMap(depth_display, cv.COLORMAP_JET)
        depth_colored[depth_image == 0] = [0, 0, 0]
        img_widget.value = to_jpeg_bytes(depth_colored)
        elapsed = time.monotonic() - t_start
        fps = frame_count / elapsed if elapsed > 0 else 0
        h, w = depth_image.shape[:2]
        center = depth_image[h // 2, w // 2]
        label.value = (f'Depth | Frame {frame_count} | {elapsed:.1f}s / {DURATION}s | '
                       f'{fps:.1f} FPS | Center: {center:.0f} cm | Range: 0-{MAX_DEPTH_CM} cm')
    time.sleep(0.05)

elapsed = time.monotonic() - t_start
rate = frame_count / elapsed if elapsed > 0 else 0
label.value = f'Depth camera complete: {frame_count} frames in {elapsed:.1f}s = {rate:.1f} FPS'
depth_received = rc.camera.get_depth_image_async() is not None

## 4. LIDAR Stream (10 seconds)

Subscribes to `/scan` (`sensor_msgs/LaserScan`). On the physical car `rc.lidar.get_num_samples()` returns 1080 (RPLIDAR with `angle_compensate=true`). The sim returns 720. Always use `get_num_samples()` instead of hard-coding either length.

In [ ]:
DURATION = 10
MAX_RANGE_CM = 1000
img_widget = widgets.Image(format='jpeg', width=320, height=320)
label = widgets.Label(value='Starting...')
display(widgets.VBox([label, img_widget]))

frame_count = 0
t_start = time.monotonic()

while time.monotonic() - t_start < DURATION:
    scan = rc.lidar.get_samples_async()
    if scan is not None and len(scan) > 0:
        frame_count += 1
        radius = 160
        image = np.zeros((2 * radius, 2 * radius, 3), np.uint8)
        n = len(scan)
        for i in range(n):
            d = scan[i]
            if 0 < d < MAX_RANGE_CM:
                angle = 2 * np.pi * i / n
                length = radius * d / MAX_RANGE_CM
                r = int(radius - length * np.cos(angle))
                c = int(radius + length * np.sin(angle))
                if 0 <= r < 2 * radius and 0 <= c < 2 * radius:
                    image[r, c, 2] = 255
        cv.circle(image, (radius, radius), 3, (0, 255, 0), -1)
        img_widget.value = to_jpeg_bytes(image, resize=None)
        forward = rc_utils.get_lidar_average_distance(scan, 0) if hasattr(rc_utils, 'get_lidar_average_distance') else float(scan[0])
        elapsed = time.monotonic() - t_start
        fps = frame_count / elapsed if elapsed > 0 else 0
        label.value = (
            f'LIDAR | Samples: {n} | Frame {frame_count} | {elapsed:.1f}s / {DURATION}s | '
            f'{fps:.1f} Hz | Forward: {forward:.0f} cm'
        )
    time.sleep(0.05)

elapsed = time.monotonic() - t_start
rate = frame_count / elapsed if elapsed > 0 else 0
label.value = f'LIDAR complete: {frame_count} scans in {elapsed:.1f}s = {rate:.1f} Hz'

## 5. IMU / Physics Data

Subscribes to `/imu` (`sensor_msgs/Imu`) and `/mag` (`sensor_msgs/MagneticField`). On the v2 physical car the IMU frame is x=front, y=right, z=up; gravity should read ~9.81 m/s² along whichever axis is currently pointing up.

In [ ]:
for i in range(5):
    accel = rc.physics.get_linear_acceleration()
    gyro = rc.physics.get_angular_velocity()
    mag = rc.physics.get_magnetic_field()

    print(f'--- Sample {i+1} ---')
    print(f'  Accel: ({accel[0]:7.2f}, {accel[1]:7.2f}, {accel[2]:7.2f}) m/s^2')
    print(f'  Gyro:  ({gyro[0]:7.3f}, {gyro[1]:7.3f}, {gyro[2]:7.3f}) rad/s')
    print(f'  Mag:   ({mag[0]:+.3e}, {mag[1]:+.3e}, {mag[2]:+.3e}) T')
    time.sleep(0.2)

accel_mag = float(np.linalg.norm(accel))
print(f'\nAccel magnitude: {accel_mag:.2f} m/s^2 (expected ~9.81 when stationary)')
if 8.0 < accel_mag < 12.0:
    print('PASS: IMU acceleration magnitude looks correct')
else:
    print('WARN: Acceleration magnitude outside expected range — check calibration')

## 6. Coral EdgeTPU Inference (10 seconds)

Subscribes to `/edgetpu/inference` (`vision_msgs/Detection2DArray`) via the new `rc.vision` module. Overlays bounding boxes on the live forward-camera frame.

In [ ]:
DURATION = 10
img_widget = widgets.Image(format='jpeg', width=640, height=480)
label = widgets.Label(value='Starting...')
display(widgets.VBox([label, img_widget]))

frame_count = 0
total_detections = 0
t_start = time.monotonic()

while time.monotonic() - t_start < DURATION:
    color_image = rc.camera.get_color_image_async()
    detections = rc.vision.get_detections_async()
    if color_image is not None:
        frame_count += 1
        annotated = color_image.copy()
        for det in detections:
            cx, cy, w, h = det.bbox
            x1, y1 = int(cx - w / 2), int(cy - h / 2)
            x2, y2 = int(cx + w / 2), int(cy + h / 2)
            cv.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv.putText(
                annotated, f'{det.class_id} {det.score:.0%}', (x1, y1 - 8),
                cv.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2,
            )
        total_detections += len(detections)
        img_widget.value = to_jpeg_bytes(annotated, resize=None)
        elapsed = time.monotonic() - t_start
        fps = frame_count / elapsed if elapsed > 0 else 0
        label.value = (
            f'EdgeTPU | Frame {frame_count} | {elapsed:.1f}s / {DURATION}s | '
            f'{fps:.1f} FPS | Detections: {len(detections)} | Total: {total_detections}'
        )
    time.sleep(0.05)

elapsed = time.monotonic() - t_start
rate = frame_count / elapsed if elapsed > 0 else 0
label.value = f'EdgeTPU complete: {frame_count} frames in {elapsed:.1f}s = {rate:.1f} FPS | Total detections: {total_detections}'

## 7. EasySMX Controller (10 seconds)

Move the joysticks, pull triggers, press buttons to verify each axis and button index. NOTE: the v1 button/axis mapping in `controller_real.py` has not yet been re-verified against the EasySMX KC-8236 in Xbox-360 fallback mode — this cell is the empirical re-map source.

In [ ]:
DURATION = 10

timer_label = widgets.Label(value='Starting...')
left_label = widgets.Label(value='Left joy:  (0.00, 0.00)')
right_label = widgets.Label(value='Right joy: (0.00, 0.00)')
trigger_label = widgets.Label(value='Triggers:  L=0.00  R=0.00')
button_label = widgets.Label(value='Buttons:   none')

display(widgets.VBox([timer_label, left_label, right_label, trigger_label, button_label]))

t_start = time.monotonic()
while time.monotonic() - t_start < DURATION:
    elapsed = time.monotonic() - t_start
    timer_label.value = f'EasySMX Controller | {elapsed:.1f}s / {DURATION}s'

    lx, ly = rc.controller.get_joystick(rc.controller.Joystick.LEFT)
    rx, ry = rc.controller.get_joystick(rc.controller.Joystick.RIGHT)
    lt = rc.controller.get_trigger(rc.controller.Trigger.LEFT)
    rt = rc.controller.get_trigger(rc.controller.Trigger.RIGHT)

    left_label.value = f'Left joy:  ({lx:+.2f}, {ly:+.2f})'
    right_label.value = f'Right joy: ({rx:+.2f}, {ry:+.2f})'
    trigger_label.value = f'Triggers:  L={lt:.2f}  R={rt:.2f}'

    pressed = [b.name for b in rc.controller.Button if rc.controller.is_down(b)]
    button_label.value = f'Buttons:   {", ".join(pressed) if pressed else "none"}'

    time.sleep(0.05)

timer_label.value = f'EasySMX Controller test complete ({DURATION}s)'

## 8. Drive Stack — mux observation (30 seconds)

Subscribes to `/mux_out` (`ackermann_msgs/AckermannDriveStamped`) so we can see exactly what the mux is forwarding to the throttle/pwm chain. The driver mux selects between `/gamepad_drive` and the student's `/drive` based on the LB/RB bumpers:

- **No bumper** → IDLE (zero)
- **LB held** → GAMEPAD (raw joystick → mux)
- **RB held** → AUTONOMY (student `/drive` → mux). This cell publishes a slow square-wave on `rc.drive` while RB is held, so you can confirm the mux is forwarding student commands.

In [ ]:
import rclpy as ros2
from ackermann_msgs.msg import AckermannDriveStamped
from rclpy.qos import QoSProfile, QoSReliabilityPolicy, QoSDurabilityPolicy

mux_node = ros2.create_node('mux_test_sub')
mux_qos = QoSProfile(depth=1)
mux_qos.reliability = QoSReliabilityPolicy.BEST_EFFORT
mux_qos.durability = QoSDurabilityPolicy.VOLATILE
mux_latest = [AckermannDriveStamped()]

def _mux_cb(msg):
    mux_latest[0] = msg

mux_node.create_subscription(AckermannDriveStamped, '/mux_out', _mux_cb, mux_qos)
rc._RacecarReal__executor.add_node(mux_node)

DURATION = 30

# Drive command amplitude. The signal goes through three multiplicative
# scalers before reaching the ESC, so a small value here disappears into the
# ESC deadband:
#   - student-side:  msg.drive.speed = command * STUDENT_MAX_SPEED  (drive_real)
#   - throttle_node: out.speed       = in.speed  * max_speed_forward (~0.5)
#   - pwm_node:      pwm = center + sign * speed * magnitude (3000 us swing)
# Net forward PWM offset = STUDENT_MAX_SPEED * STUDENT_AMPLITUDE * 0.5 * 3000.
# An ESC needs roughly >= +/-300 us off center before the motor starts to
# turn, so the inputs below give ~+/-450 us — comfortably past the deadband
# but still gentle for a benchtop test. Drop these if the car has tires off
# the ground and you want to see motor spin without rolling.
STUDENT_MAX_SPEED = 0.5
STUDENT_AMPLITUDE = 0.6
STUDENT_STEERING  = 0.4

timer_label = widgets.Label(value='Starting...')
bumper_label = widgets.Label(value='Bumpers: ...')
mode_label = widgets.Label(value='Mux mode: ...')
output_label = widgets.Label(value='Mux output: ...')
student_label = widgets.Label(value='Student /drive: ...')

display(widgets.VBox([timer_label, bumper_label, mode_label, student_label, output_label]))

rc.drive.set_max_speed(STUDENT_MAX_SPEED)
rc.drive.stop()

t_start = time.monotonic()
last_mux_stamp = 0.0
while time.monotonic() - t_start < DURATION:
    elapsed = time.monotonic() - t_start
    timer_label.value = f'Drive / Mux | {elapsed:.1f}s / {DURATION}s'

    lb = rc.controller.is_down(rc.controller.Button.LB)
    rb = rc.controller.is_down(rc.controller.Button.RB)
    bumper_label.value = f'Bumpers:   LB={"HELD" if lb else "---"}  RB={"HELD" if rb else "---"}'

    if rb and not lb:
        phase = (elapsed % 3.0) < 1.5
        speed = STUDENT_AMPLITUDE if phase else -STUDENT_AMPLITUDE
        angle = STUDENT_STEERING  if phase else -STUDENT_STEERING
        rc.drive.set_speed_angle(speed, angle)
        mode_label.value = 'Mux mode:  AUTONOMY (RB) — student /drive forwarded'
        # Show both the raw command and what hits /drive after the student
        # library's max_speed scaling, so the wheels-not-moving case is
        # unambiguous (raw 0.6 vs published 0.3 vs mux-out 0.3 vs PWM offset).
        published_speed = speed * STUDENT_MAX_SPEED
        student_label.value = (
            f'Student /drive: raw=({speed:+.2f}, {angle:+.2f})  '
            f'after max_speed={STUDENT_MAX_SPEED}: speed={published_speed:+.3f}'
        )
    elif lb and not rb:
        rc.drive.stop()
        mode_label.value = 'Mux mode:  GAMEPAD (LB) — /gamepad_drive forwarded'
        student_label.value = 'Student /drive: stopped (mux is on the gamepad source)'
    else:
        rc.drive.stop()
        mode_label.value = 'Mux mode:  IDLE — zero out'
        student_label.value = 'Student /drive: stopped'

    m = mux_latest[0]
    mux_age = elapsed - last_mux_stamp if last_mux_stamp else 0.0
    if m.header.stamp.sec or m.header.stamp.nanosec:
        last_mux_stamp = elapsed
    output_label.value = (
        f'Mux out:   speed={m.drive.speed:+.3f}  steering_angle={m.drive.steering_angle:+.3f}  '
        f'(stamp_sec={m.header.stamp.sec})'
    )
    time.sleep(0.05)

rc.drive.stop()
timer_label.value = f'Drive / Mux test complete ({DURATION}s)'
rc._RacecarReal__executor.remove_node(mux_node)
mux_node.destroy_node()

## 9. Battery Power — Voltage & Current

Reads the NEO-PIT INA226 power sensor through the physics module:
`rc.physics.get_battery_voltage()` (volts) and `rc.physics.get_battery_current()` (amps).
`pit_node` republishes the Teensy telemetry on `/battery/voltage` and `/battery/current`.
Voltage is the main pack bus voltage; current is the draw on that rail (light when only the Teensy circuit is powered).

In [ ]:
DURATION = 8

timer_label = widgets.Label(value='Starting...')
volt_label = widgets.Label(value='Voltage: ...')
curr_label = widgets.Label(value='Current: ...')
display(widgets.VBox([timer_label, volt_label, curr_label]))

v_min, v_max = 1e9, -1e9
t_start = time.monotonic()
while time.monotonic() - t_start < DURATION:
    v = rc.physics.get_battery_voltage()
    c = rc.physics.get_battery_current()
    v_min, v_max = min(v_min, v), max(v_max, v)
    elapsed = time.monotonic() - t_start
    timer_label.value = f'Battery power | {elapsed:.1f}s / {DURATION}s'
    volt_label.value = f'Voltage: {v:6.2f} V   (min {v_min:.2f}, max {v_max:.2f})'
    curr_label.value = f'Current: {c:6.2f} A'
    time.sleep(0.1)

v = rc.physics.get_battery_voltage()
c = rc.physics.get_battery_current()
power_ok = 5.0 < v < 13.0 and c >= 0.0
print(f'Final: {v:.2f} V, {c:.2f} A')
print('PASS: voltage in [5, 13] V and current >= 0' if power_ok
      else 'WARN: voltage/current out of expected range — check the pack and power sensor')

## 10. FlySky RC Controller

Reads the 8-channel FlySky iA6B receiver through `rc.physics.get_rc_channels()`, which returns eight values normalized to `[-1, 1]` (0 at center / no signal). Turn the transmitter ON and move both sticks and the switches.

Channel map (verified on hardware):

| idx | control | idx | control |
|-----|---------|-----|---------|
| 0 | right stick X | 4 | switch A |
| 1 | right stick Y | 5 | switch B |
| 2 | left stick Y (throttle) | 6 | switch C |
| 3 | left stick X | 7 | switch D |

In [ ]:
DURATION = 10
CH_NAMES = ['0 right-X', '1 right-Y', '2 left-Y (throttle)', '3 left-X',
            '4 switch-A', '5 switch-B', '6 switch-C', '7 switch-D']

timer_label = widgets.Label(value='Starting... (turn the transmitter ON, move sticks + switches)')
ch_labels = [widgets.Label(value=f'{n}: 0.00') for n in CH_NAMES]
display(widgets.VBox([timer_label] + ch_labels))

mins = [9.0] * 8
maxs = [-9.0] * 8
t_start = time.monotonic()
while time.monotonic() - t_start < DURATION:
    ch = rc.physics.get_rc_channels()               # 8 values, each normalized to [-1, 1]
    for i in range(8):
        v = float(ch[i])
        mins[i] = min(mins[i], v)
        maxs[i] = max(maxs[i], v)
        ch_labels[i].value = f'{CH_NAMES[i]}: {v:+.2f}   (min {mins[i]:+.2f}, max {maxs[i]:+.2f})'
    elapsed = time.monotonic() - t_start
    timer_label.value = f'FlySky RC | {elapsed:.1f}s / {DURATION}s'
    time.sleep(0.05)

moved = sum(1 for i in range(8) if (maxs[i] - mins[i]) > 0.3)
rc_ok = moved > 0
print(f'{moved} channel(s) swung more than 0.3 during the window.')
print('PASS: transmitter detected and channels responding' if rc_ok
      else 'WARN: no channel moved — is the transmitter on and bound?')

## 11. Encoder Speed

`rc.physics.get_encoder_speed()` returns the car's forward speed in m/s, derived from the NEO-PIT hall encoder (the Teensy applies the gear ratios and wheel circumference). The library does not expose raw motor rpm, so this cell also derives an approximate **wheel** rpm from the wheel circumference for reference. Spin a wheel by hand or drive the car to see it change.

In [ ]:
DURATION = 10
WHEEL_DIAMETER_M = 0.072                    # NEO-PIT wheel (firmware CAR_SPECS)
WHEEL_CIRC_M = np.pi * WHEEL_DIAMETER_M

timer_label = widgets.Label(value='Starting... (spin a wheel by hand or drive)')
speed_label = widgets.Label(value='Speed: ...')
rpm_label = widgets.Label(value='Wheel rpm: ...')
display(widgets.VBox([timer_label, speed_label, rpm_label]))

max_abs = 0.0
t_start = time.monotonic()
while time.monotonic() - t_start < DURATION:
    speed = rc.physics.get_encoder_speed()          # m/s, gear ratios applied on the Teensy
    wheel_rpm = (speed / WHEEL_CIRC_M) * 60.0        # derived reference (library exposes m/s, not raw rpm)
    max_abs = max(max_abs, abs(speed))
    elapsed = time.monotonic() - t_start
    timer_label.value = f'Encoder | {elapsed:.1f}s / {DURATION}s'
    speed_label.value = f'Speed: {speed:+.3f} m/s   (max |v| {max_abs:.3f})'
    rpm_label.value = f'Wheel rpm: {wheel_rpm:+.1f}'
    time.sleep(0.05)

encoder_ok = max_abs > 0.0
print(f'Peak speed seen: {max_abs:.3f} m/s')
print('PASS: encoder responded to motion' if encoder_ok
      else 'INFO: no motion detected — spin a wheel during the window to confirm the encoder')

## 12. Dot-matrix Display

The v2 student library publishes to `/dotmatrix/pixels` (`std_msgs/UInt8MultiArray`) and `/dotmatrix/text` (`std_msgs/String`) so it cooperates with the driver's `dotmatrix_node` instead of fighting it for the SPI bus. This cell sends two patterns and a scrolling text.

In [ ]:
# Pattern A: solid bar across the bottom row
bar = rc.display.new_matrix()
bar[7, :] = 1
rc.display.set_matrix(bar)
print('Sent bottom-row bar. Holding 2 s...')
time.sleep(2)

# Pattern B: diagonal sweep
diag = rc.display.new_matrix()
for i in range(min(diag.shape[0], diag.shape[1])):
    diag[i, i] = 1
    diag[i, diag.shape[1] - 1 - i] = 1
rc.display.set_matrix(diag)
print('Sent diagonal cross. Holding 2 s...')
time.sleep(2)

# Pattern C: scrolling text via /dotmatrix/text
rc.display.show_text('RACECAR NEO V2')
print('Sent scrolling text via /dotmatrix/text')
time.sleep(10)

# Clear
rc.display.set_matrix(rc.display.new_matrix())
dotmatrix_ok = True
print('Cleared.')

## 13. LED Strip

`rc.led` publishes per-pixel RGB to `/led/pixels`; `pit_node` forwards it to the Teensy's WS2812B strip (84 pixels). This cell cycles solid colors, a rainbow gradient, and a single-pixel chase, then clears. Colors are `(r, g, b)`, 0-255.

In [ ]:
import colorsys

N = rc.led.get_num_pixels()
print(f'LED strip: {N} pixels')

for name, color in [('RED', (255, 0, 0)), ('GREEN', (0, 255, 0)),
                    ('BLUE', (0, 0, 255)), ('WHITE', (255, 255, 255))]:
    rc.led.set_color(color)
    print(f'All {name}. Holding 1.5 s...')
    time.sleep(1.5)

# Rainbow gradient across the strip
gradient = []
for i in range(N):
    r, g, b = colorsys.hsv_to_rgb(i / N, 1.0, 1.0)
    gradient.append((int(r * 255), int(g * 255), int(b * 255)))
rc.led.set_pixels(gradient)
print('Rainbow gradient. Holding 3 s...')
time.sleep(3)

# Single-pixel chase
print('Single-pixel chase...')
for i in range(N):
    rc.led.clear()
    rc.led.set_pixel(i, (255, 255, 255))
    time.sleep(0.02)

rc.led.clear()
led_ok = True
print('Cleared. LED strip test complete.')

## 14. Telemetry — record and visualize

`rc.telemetry.declare_variables(...)` opens a timestamped CSV under `labs/logs/`; `record(...)` appends a row; `visualize()` writes a sibling PNG. We log accel magnitude and a synthetic sine for 5 seconds and then plot.

In [ ]:
rc.telemetry.declare_variables('accel_mag', 'sine')

DURATION = 5
t_start = time.monotonic()
n = 0
while time.monotonic() - t_start < DURATION:
    t = time.monotonic() - t_start
    a = rc.physics.get_linear_acceleration()
    rc.telemetry.record(float(np.linalg.norm(a)), float(np.sin(2 * np.pi * 0.5 * t)))
    n += 1
    time.sleep(0.05)

rc.telemetry.visualize()
print(f'Logged {n} telemetry samples in {DURATION}s.')
print(f'CSV: {rc.telemetry._LOG_FILE_NAME}')
print(f'PNG: {rc.telemetry._PLOT_FILE_NAME}')

## 15. Diagnostics

`/diagnostics` is published by `edgetpu_node` (inference timing, frame count, device health) and any other diagnostics-emitting driver nodes. `rc.telemetry.get_diagnostics()` returns a dict keyed by status name.

In [ ]:
diag = rc.telemetry.get_diagnostics()

if diag:
    for name, info in diag.items():
        level = info['level']
        if isinstance(level, (bytes, bytearray)):
            level = int.from_bytes(level, byteorder='big')
        level_str = {0: 'OK', 1: 'WARN', 2: 'ERROR', 3: 'STALE'}.get(level, f'LVL:{level}')
        print(f'[{level_str}] {name}: {info["message"]}')
        for k, v in info.items():
            if k not in ('level', 'message', 'hardware_id'):
                print(f'    {k}: {v}')
        print()
else:
    print('No diagnostics received yet. Confirm edgetpu_enable:=true on the teleop launch.')

## 16. Summary

In [ ]:
color = rc.camera.get_color_image_async()
scan = rc.lidar.get_samples_async()
accel = rc.physics.get_linear_acceleration()
diagnostics = rc.telemetry.get_diagnostics()
detections = rc.vision.get_detections_async()
voltage = rc.physics.get_battery_voltage()
current = rc.physics.get_battery_current()
rc_channels = rc.physics.get_rc_channels()

results = {
    'Forward camera (/camera/color)': color is not None,
    'Depth stream (/camera/depth)': rc.camera.get_depth_image_async() is not None,
    'LIDAR (/scan)': scan is not None and len(scan) > 0,
    'IMU (/imu/fused, /mag)': bool(np.any(accel != 0)),
    'Controller (/joy)': rc.controller.get_joystick(rc.controller.Joystick.LEFT) is not None,
    'Drive publisher (/drive)': True,
    'Mux observation (/mux_out)': True,
    'Battery power (/battery/voltage, /battery/current)': 5.0 < voltage < 13.0 and current >= 0.0,
    'FlySky RC (/rc/channels)': rc_channels is not None and len(rc_channels) == 8,
    'Encoder (/encoder/speed)': rc.physics.get_encoder_speed() is not None,
    'Dot-matrix (/dotmatrix/pixels, /dotmatrix/text)': True,
    'LED strip (/led/pixels)': rc.led.get_num_pixels() > 0,
    'Telemetry (CSV + PNG + /diagnostics)': rc.telemetry._LOG_FILE_NAME is not None,
    'EdgeTPU vision (/edgetpu/inference)': len(detections) >= 0,
    'Diagnostics (/diagnostics)': len(diagnostics) > 0,
}

print('=' * 64)
print('  RACECAR Neo v2 — Async Core Test Results')
print('=' * 64)
all_pass = True
for name, ok in results.items():
    status = 'PASS' if ok else 'FAIL'
    if not ok:
        all_pass = False
    print(f'  [{status}] {name}')
print('=' * 64)
print(f'  Overall: {"ALL PASS" if all_pass else "SOME FAILURES — check above"}')
print('=' * 64)